# LandIQ 2018 — Tomato vs Non-Tomato (binary labels)

This notebook counts:
- **Tomato polygons**: where ANY of `CROPTYP1/2/3` contains `T15` or `T26`.
- **Non-tomato polygons**: everything else, excluding any row that contains tomato codes in any slot.

It also summarizes **non-tomato** negatives by a coarse DWR group derived from the leading code letter (e.g. `F*` → `F`, `C*` → `C`, `YP` → `YP`, `****` → `UNK`).


In [ ]:
from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import pandas as pd
import yaml

from src.landiq.filter_non_tomato import dwr_group_from_code
from src.landiq.legend_codes import tomato_mask_any_croptyp
from src.utils.paths import REPO_ROOT, resolve_landiq_shapefile_path

cfg_path = REPO_ROOT / "configs" / "paths.local.yaml"
if not cfg_path.is_file():
    cfg_path = REPO_ROOT / "configs" / "paths.example.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
lc = cfg["landiq"]

in_path = resolve_landiq_shapefile_path(cfg, REPO_ROOT)
gdf = gpd.read_file(in_path)

crop_columns = lc.get("crop_columns")
if not crop_columns:
    c = lc.get("crop_column")
    crop_columns = [c] if c else []

if not crop_columns:
    raise ValueError("Set landiq.crop_columns (recommended) or landiq.crop_column")

tomato_values = lc.get("tomato_values") or []
if not tomato_values:
    raise ValueError("Set landiq.tomato_values (e.g. [\"T15\", \"T26\"]) in configs/paths.local.yaml")

tomato_mask = tomato_mask_any_croptyp(gdf, tomato_values, columns=crop_columns)
tomato_n = int(tomato_mask.sum())
non_n = len(gdf) - tomato_n

print("Input:", in_path)
print("Total polygons:", len(gdf))
print("Tomato polygons:", tomato_n, "| Non-tomato polygons:", non_n)

non_gdf = gdf.loc[~tomato_mask].copy()
tomato_set = {str(v).strip() for v in tomato_values}

def group_for_row(row: pd.Series) -> str:
    for col in crop_columns:
        if col not in row:
            continue
        v = row[col]
        if v is None:
            continue
        s = str(v).strip()
        if not s or s in tomato_set:
            continue
        return dwr_group_from_code(s)
    return "UNK"

non_groups = non_gdf.apply(group_for_row, axis=1)

print("\nNon-tomato groups (coarse DWR):")
print(non_groups.value_counts().sort_values(ascending=False).to_string())

print("\nNon-tomato exact CROPTYP code distribution (top 25):")
codes: list[str] = []
for col in crop_columns:
    codes.extend(non_gdf[col].dropna().astype(str).str.strip().tolist())
code_series = pd.Series(codes)
code_series = code_series[~code_series.isin(tomato_set)]
print(code_series.value_counts().head(25).to_string())
